In [24]:
import torch  # torch.tensor
import torch.nn as nn  # nn.Module, nn.Linear
import torch.nn.functional as F  # 为了使用softmax()函数.

In [25]:
# 新建一个"自注意力"类
class SelfAttention(nn.Module):
    def __init__(self, d_model=2, row_dim=0, col_dim=1):
        # 调用父类的方法
        super().__init__()

        # 初始化q,k,v的权重矩阵
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        # 为了方便后面使用
        self.row_dim = row_dim
        self.col_dim = col_dim

    # token_encodings的维度: 3 * 2
    # 3个单词, 每个单词的词向量空间是2维的.
    def forward(self, token_encodings):
        q = self.W_q(token_encodings)
        k = self.W_k(token_encodings)
        v = self.W_v(token_encodings)

        # 分步实现attention分数的计算.

        # q's dimension -> 3*2, k.transpose's dimension -> 2 * 3
        # sims's dimension -> 2 * 2
        # 计算相似性
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))

        # 重要的是词向量的维度
        # 归一化
        scaled_sim = sims / torch.tensor(k.size(self.col_dim)**0.5)

        attention_percents = F.softmax(scaled_sim, dim=self.col_dim)

        attention_scores = torch.matmul(attention_percents, v)

        # 返回注意力分数
        return attention_scores

In [26]:
encodings_matrix = torch.tensor([[1.16, 0.23],
                                 [0.57, 1.36],
                                 [4.41, -2.16]])

# 设置随机数种子
torch.manual_seed(42)

# selfAttention = SelfAttention(d_model=2, row_dim=0, col_dim=1)
selfAttention = SelfAttention()

selfAttention(encodings_matrix)

tensor([[1.0100, 1.0641],
        [0.2040, 0.7057],
        [3.4989, 2.2427]], grad_fn=<MmBackward0>)

In [27]:
# 看看q,k,v的权重参数,线性层的权重参数
# 矩阵大小是2*2
print(type(selfAttention.W_q.weight))
print(f"W_q: {selfAttention.W_q.weight}")
print(f"W_k: {selfAttention.W_k.weight}")
print(f"W_v: {selfAttention.W_v.weight}")

<class 'torch.nn.parameter.Parameter'>
W_q: Parameter containing:
tensor([[ 0.5406,  0.5869],
        [-0.1657,  0.6496]], requires_grad=True)
W_k: Parameter containing:
tensor([[-0.1549,  0.1427],
        [-0.3443,  0.4153]], requires_grad=True)
W_v: Parameter containing:
tensor([[ 0.6233, -0.5188],
        [ 0.6146,  0.1323]], requires_grad=True)


In [29]:
selfAttention.W_k.weight.transpose(0, 1)

tensor([[-0.1549, -0.3443],
        [ 0.1427,  0.4153]], grad_fn=<TransposeBackward0>)